In [0]:
#Notebook 03: Raw to Bronze


#Widgets
dbutils.widgets.text("catalog_name", "chinook_catalog")
dbutils.widgets.text("meta_schema", "pipeline_metadata")
dbutils.widgets.text("bronze_schema", "bronze")

catalog_name  = dbutils.widgets.get("catalog_name")
meta_schema   = dbutils.widgets.get("meta_schema")
bronze_schema = dbutils.widgets.get("bronze_schema")

#Fetching the successfully written files from child metadata for the most recent run
metrics_df = spark.sql(f"""
    SELECT table_name, file_location
    FROM {catalog_name}.{meta_schema}.child_metrics_metadata
    WHERE status = 'SUCCESS'
    AND created_date = current_date()
""")

tables = metrics_df.collect()
print(f"Tables to load into Bronze: {[row['table_name'] for row in tables]}")

#Looping through each table
for row in tables:
    table_name    = row["table_name"]
    file_location = row["file_location"]

    try:
        #Reading Parquet files from Raw Volume
        raw_df = spark.read.parquet(file_location)
        raw_row_count = raw_df.count()

        #Overwriting delta to Bronze to ensure we are alays laoding today's snapshot
        raw_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.{bronze_schema}.{table_name.lower()}")

        #Verifying the write to Bronze
        bronze_df = spark.sql(f"""
            SELECT * FROM {catalog_name}.{bronze_schema}.{table_name.lower()}
        """)
        bronze_row_count = bronze_df.count()

        status = "Successful" if raw_row_count == bronze_row_count else "Unsuccessful"
        print(f"{status} {table_name} | Raw Row Count: {raw_row_count} | Bronze Row Count: {bronze_row_count}")

    except Exception as e:
        print(f"{table_name} failed: {str(e)}")

print("Write from Raw to Bronze completed...")

In [0]:
%sql
SELECT * FROM chinook_catalog.bronze.playlist;

In [0]:
%sql
SELECT DISTINCT PlaylistId FROM chinook_catalog.bronze.playlist;

In [0]:
df = spark.sql("select * from chinook_catalog.bronze.track")
df.printSchema()

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS chinook_catalog.silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS chinook_catalog.gold